In [1]:
import scanpy as sc
combined = sc.read_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/filter_defined_number_combined_sampled.h5ad")

/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel fro

In [4]:
print("原始数据中的样本分布：")
print(combined.obs['orig.ident'].value_counts())

原始数据中的样本分布：
orig.ident
Kimberly Siletti    232115
Braun               169748
zeng                 13774
Ai                    8675
mole                  5229
zhou                  4244
tyser                 1189
Petropoulos_2016      1064
xiang                  521
Yanagida_2021          202
Yuan_2025              162
Yan_2013                89
Xue_2013                28
Name: count, dtype: int64


In [3]:
import pandas as pd
import numpy as np

In [6]:
samples_to_downsample = ['Braun', 'Kimberly Siletti']
target_fraction = 0.2  # 十分之一

# 初始化一个布尔数组，标记所有行初始都不被选中
sample_mask = np.zeros(combined.n_obs, dtype=bool)

# 对每个需要降采样的样本单独处理
for sample in samples_to_downsample:
    # 获取当前样本的所有细胞索引
    sample_indices = combined.obs[combined.obs['orig.ident'] == sample].index
    n_cells_sample = len(sample_indices)
    
    # 计算当前样本需要抽取的细胞数量
    n_sample_to_keep = int(n_cells_sample * target_fraction)
    
    # 如果计算出的数量为0，则至少保留1个，除非原样本本身就是空的
    if n_sample_to_keep == 0 and n_cells_sample > 0:
        n_sample_to_keep = 1
    
    # 从当前样本中随机抽取指定数量的细胞
    # 使用random.choice，设置replace=False确保无放回抽样
    rng = np.random.default_rng(seed=42)  # 设置随机种子以保证结果可重现
    selected_indices = rng.choice(sample_indices, size=n_sample_to_keep, replace=False)
    
    # 将选中的细胞在掩码中标记为True
    sample_mask[combined.obs.index.get_indexer(selected_indices)] = True

# 对于不需要降采样的样本，保留所有细胞
other_samples_mask = ~combined.obs['orig.ident'].isin(samples_to_downsample)
sample_mask |= other_samples_mask  # 合并掩码

# 4. 应用采样
adata_downsampled = combined[sample_mask, :].copy()

# 5. 检查采样后的样本分布
print("\n降采样后的样本分布：")
print(adata_downsampled.obs['orig.ident'].value_counts())




降采样后的样本分布：
orig.ident
Kimberly Siletti    46423
Braun               33949
zeng                13774
Ai                   8675
mole                 5229
zhou                 4244
tyser                1189
Petropoulos_2016     1064
xiang                 521
Yanagida_2021         202
Yuan_2025             162
Yan_2013               89
Xue_2013               28
Name: count, dtype: int64


In [9]:
del adata_downsampled.obs['lineage']

In [11]:
adata_downsampled

AnnData object with n_obs × n_vars = 115549 × 29772
    obs: 'stage', 'nCount_RNA', 'nFeature_RNA', 'orig_anno', 'orig_sub_anno', 'sample', 'percent.mt', 'reanno', 'orig.ident', 'dataset'
    obsm: 'X_umap'
    layers: 'counts'

In [10]:
# 6. 保存新的数据集
# 替换成你希望保存的输出路径
adata_downsampled.write('/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/data/3sample_smalldataset_test_script_model_nolineage.h5ad')

print("操作完成！新的数据集已保存。")

操作完成！新的数据集已保存。
